In [1]:
import pint
import json
import jsonata
from IPython.display import JSON, HTML
import pandas as pd

# ── ANSI palette ──────────────────────────────────────────
RESET, BOLD, DIM = "\033[0m", "\033[1m", "\033[2m"
CYAN, GREEN, YELLOW = "\033[36m", "\033[32m", "\033[33m"

reg = pint.UnitRegistry()
reg.define('kg = kilogram')
reg.define('mg = milligram')
reg.define('mcg = microgram')
reg.define('mL = milliliter')
reg.define('ug = microgram')
reg.define('ng = nanogram')
reg.define('cg = centigram')
reg.define('dL = deciliter')
reg.define('L = liter')
reg.define('g = gram')
reg.define('gm = gram')
reg.define('equivalent = [substance_charge] = Eq')
reg.define('meq = 0.001 equivalent = mEq')
reg.define('mEq = 0.001 equivalent = meq')
# Activity / arbitrary units family — all dimensionally [activity]
reg.define('unit = [activity] = U = IU = iu = USP_U = arb_U')
reg.define('milliunit = 0.001 unit = mU = mIU')
reg.define('million_unit = 1e6 unit = MU = MIU')
mass = 1000 * reg.mg
vol = 1000 * reg.mL
conc = mass / vol

UNITS = ("mg/mL", "cg/dL", "ug/mL", "ng/mL", "g/mL", "mg/L", "ug/L", "g/L")


def show_conc(conc, units=UNITS, prec=6):
      """Print a concentration in multiple units as an aligned, colorized table."""
      print(f"{BOLD}{CYAN}CONC:{RESET} {BOLD}{conc:~P}{RESET}")
      w = max(map(len, units))
      for u in units:
            q = conc.to(u)
            print(f"-> {GREEN}{q.magnitude:>7,.{prec}g}{RESET}"
                  f" {RESET} {YELLOW}{u:<{w}}{RESET}")


show_conc(conc)

CONC: 1.0 mg/mL
->       1  mg/mL
->      10  cg/dL
->   1,000  ug/mL
->   1e+06  ng/mL
->   0.001  g/mL 
->   1,000  mg/L 
->   1e+06  ug/L 
->       1  g/L  


## FDA

## Functions

In [2]:
import re
import pint
from dataclasses import dataclass
from fda import clean_labeler_name

_STRENGTH_RE = re.compile(
      r'^\s*'
      r'(?P<num_val>\d+(?:\.\d+)?)\s*(?P<num_unit>[a-zA-Zµμ%]+)'
      r'\s*(?:/\s*'
      r'(?P<den_val>\d+(?:\.\d+)?)?\s*(?P<den_unit>[a-zA-Zµμ]+)'
      r')?\s*$'
)
_UCUM_BRACKET_RE = re.compile(r"\[([^\]]+)\]")
# language=JSONata
_JSONATA_PTYPE_EXTRACTOR = r"$ ~> |**[description]|{'product_type': $reverse($match(description, /\b[A-Z\s,\-]{3,}/).match)[0]}|"


def extract_product_type(obj):
      return jsonata.Jsonata(_JSONATA_PTYPE_EXTRACTOR).evaluate(obj)


def _normalize_ucum(s: str) -> str:
      # [USP'U] -> USP_U, [iU] -> iU, [arb'U] -> arb_U
      return _UCUM_BRACKET_RE.sub(
            lambda m: m.group(1).replace("'", "_").replace(".", "_"),
            s,
      )


def _normalize_denom(s: str) -> str:
      # ".9 g/100mL" -> ".9 g/(100mL)"
      return re.sub(r'/\s*(\d*\.?\d+)\s*([a-zA-Z]+)', r'/(\1\2)', s)


def parse_strength(strength_str: str) -> pint.Quantity[pint.Unit]:
      """
      Parse a pharmaceutical strength string into a pint Quantity.
      """
      if not isinstance(strength_str, str) or not strength_str.strip():
            raise ValueError(f"Empty or non-string strength: {strength_str!r}")

      s_str = strength_str.strip().replace('μ', 'u').replace('µ', 'u')  # normalize micro
      s_norm_ucum = _normalize_ucum(s_str)
      s_norm = _normalize_denom(s_norm_ucum)

      m = _STRENGTH_RE.match(s_norm)
      if not m:
            # Fall back to letting pint try directly (handles odd cases pint knows about)
            try:
                  return reg.parse_expression(s_norm)
            except Exception as e:
                  raise ValueError(f"Could not parse strength {strength_str!r}: {e}") from e

      num_val = float(m.group('num_val'))
      num_unit = m.group('num_unit')
      den_unit = m.group('den_unit')
      den_val = float(m.group('den_val')) if m.group('den_val') else 1

      try:
            numerator = num_val * reg.parse_expression(num_unit)
      except pint.errors.UndefinedUnitError as e:
            raise ValueError(f"Unknown numerator unit in {strength_str!r}: {num_unit}") from e

      if den_unit is None:
            return numerator

      try:
            denominator = den_val * reg.parse_expression(den_unit)
      except pint.errors.UndefinedUnitError as e:
            raise ValueError(f"Unknown denominator unit in {strength_str!r}: {den_unit}") from e

      if denominator.magnitude == 0:
            raise ValueError(f"Zero denominator in {strength_str!r}")

      return round(numerator / denominator, 6)


def parse_strength_in_context(obj):
      if isinstance(obj, dict):
            return {
                  k: (parse_strength(v) if k == 'strength' and isinstance(v, str)
                      else parse_strength_in_context(v))
                  for k, v in obj.items()
            }
      if isinstance(obj, list):
            return [parse_strength_in_context(x) for x in obj]
      return obj


def get_product_type(packaging):
      try:
            return packaging[0]['product_type']
      except (IndexError, KeyError, TypeError):
            return None


PACKAGE_PATTERN = re.compile(
      r"/\s*"
      r"(?P<package_size>\d+(?:\.\d+)?\s*[A-Za-zµμ]+)"
      r"\s+in\s+"
      r"(?P<package_count>\d+)\s+"
      r"(?P<package_type>[A-Za-z][A-Za-z,\- ]*?)"
      r"(?=\s*(?:\(|$))",
      re.IGNORECASE,
)


@dataclass
class PackageInfo:
      package_count: int
      package_size: str
      package_type: str


def extract_package_info(value: str) -> PackageInfo | None:
      match = PACKAGE_PATTERN.search(value)

      if not match:
            return None

      return PackageInfo(
            package_count=int(match.group("package_count")),
            package_size=" ".join(match.group("package_size").split()),
            package_type=" ".join(match.group("package_type").split()),
      )


## Usage

### Potassium Phosphates

In [3]:
from rxocrpl.fda import lookup_generic_name, lookup_ndc_package

kpo4 = lookup_generic_name('potassium phosphates', dosage_form='INJECTION', fetch_rxcui=True)
JSON(kpo4)

<IPython.core.display.JSON object>

In [4]:
df = pd.DataFrame(kpo4)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,packaged_as,package_type,package_size
0,Caplin Steriles Limited,POTASSIUM PHOSPHATES,POTASSIUM PHOSPHATES,65145-195,"[{'name': 'POTASSIUM PHOSPHATE, DIBASIC', 'str...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '65145-195-25', 'description'...","{'concept': [['1928569', 'SCD']], 'drug': [['1...",generic_name,"VIAL, SINGLE-DOSE","{'packageCount': 1, 'packageSize': '15 mL', 'p...","VIAL, SINGLE-DOSE",15 mL
1,Caplin Steriles Limited,POTASSIUM PHOSPHATES,POTASSIUM PHOSPHATES,65145-194,"[{'name': 'POTASSIUM PHOSPHATE, DIBASIC', 'str...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '65145-194-25', 'description'...","{'concept': [['1928567', 'SCD']], 'drug': [['1...",generic_name,"VIAL, SINGLE-DOSE","{'packageCount': 1, 'packageSize': '5 mL', 'pa...","VIAL, SINGLE-DOSE",5 mL
2,Caplin Steriles Limited,POTASSIUM PHOSPHATES,POTASSIUM PHOSPHATES,65145-196,"[{'name': 'POTASSIUM PHOSPHATE, DIBASIC', 'str...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '65145-196-25', 'description'...","{'concept': [['2667657', 'SCD']], 'drug': [['2...",generic_name,"VIAL, PHARMACY BULK PACKAGE","{'packageCount': 1, 'packageSize': '50 mL', 'p...","VIAL, PHARMACY BULK PACKAGE",50 mL
3,"Fresenius Kabi USA, LLC",Potassium Phosphates,Potassium Phosphates,65219-656,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-656-10', 'description'...","{'concept': [['2716829', 'SCD']], 'drug': [['2...",generic_name,POUCH,"{'packageCount': 1, 'packageSize': '1 BAG', 'p...",POUCH,1 BAG
4,"Fresenius Kabi USA, LLC",Potassium Phosphates,Potassium Phosphates,65219-658,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-658-25', 'description'...","{'concept': [['2688947', 'SCD']], 'drug': [['2...",generic_name,POUCH,"{'packageCount': 1, 'packageSize': '1 BAG', 'p...",POUCH,1 BAG
5,"Hospira, Inc.",Potassium Phosphates,potassium phosphates,0409-0032,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '0409-0032-01', 'description'...","{'concept': [['1928569', 'SCD']], 'drug': [['1...",generic_name,"VIAL, SINGLE-DOSE","{'packageCount': 1, 'packageSize': '15 mL', 'p...","VIAL, SINGLE-DOSE",15 mL
6,"GLENMARK PHARMACEUTICALS INC., USA",POTASSIUM PHOSPHATES,POTASSIUM PHOSPHATES,68462-788,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '68462-788-25', 'description'...","{'concept': [['2667657', 'SCD']], 'drug': [['2...",generic_name,"VIAL, PHARMACY BULK PACKAGE","{'packageCount': 1, 'packageSize': '50 mL', 'p...","VIAL, PHARMACY BULK PACKAGE",50 mL
7,"GLENMARK PHARMACEUTICALS INC., USA",POTASSIUM PHOSPHATES,POTASSIUM PHOSPHATES,68462-786,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '68462-786-25', 'description'...","{'concept': [['1928567', 'SCD']], 'drug': [['1...",generic_name,"VIAL, SINGLE-DOSE","{'packageCount': 1, 'packageSize': '5 mL', 'pa...","VIAL, SINGLE-DOSE",5 mL
8,"GLENMARK PHARMACEUTICALS INC., USA",POTASSIUM PHOSPHATES,POTASSIUM PHOSPHATES,68462-787,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '68462-787-25', 'description'...","{'concept': [['1928569', 'SCD']], 'drug': [['1...",generic_name,"VIAL, SINGLE-DOSE","{'packageCount': 1, 'packageSize': '15 mL', 'p...","VIAL, SINGLE-DOSE",15 mL
9,Cipla USA Inc.,Potassium Phosphates,Potassium Phosphates,69097-055,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '69097-055-97', 'description'...","{'concept': [['2667657', 'SCD']], 'drug': [['2...",generic_name,VIAL,"{'packageCount': 1, 'package

In [5]:
# language=JSONata
data = jsonata.Jsonata("""
    *.packaging[$contains(description, 'BAG')].{
        'labeler_name': %.labeler_name,
        'brand_name': %.brand_name,
        'generic_name': %.generic_name,
        'product_ndc': %.product_ndc,
        'active_ingredients': %.active_ingredients,
        'package_ndc': package_ndc,
        'description': description
    }
""").evaluate(kpo4)
df = pd.DataFrame(data)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,package_ndc,description
0,"Fresenius Kabi USA, LLC",Potassium Phosphates,Potassium Phosphates,65219-656,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...",65219-656-10,24 POUCH in 1 CARTON (65219-656-10) / 1 BAG i...
1,"Fresenius Kabi USA, LLC",Potassium Phosphates,Potassium Phosphates,65219-658,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...",65219-658-25,30 POUCH in 1 CARTON (65219-658-25) / 1 BAG i...
2,Amneal Pharmaceuticals LLC,POTASSIUM PHOSPHATES,Potassium Phosphates in sodium chloride,70121-1722,"[{'name': 'POTASSIUM PHOSPHATE, DIBASIC', 'str...",70121-1722-9,24 POUCH in 1 CARTON (70121-1722-9) / 1 BAG i...


In [6]:
# language=JSONata
print('BAGS:', jsonata.Jsonata("packaging[$contains(description, 'BAG')] ~> $count()").evaluate(kpo4))
# language=JSONata
print('VIALS:', jsonata.Jsonata("packaging[$contains(description, 'VIAL')] ~> $count()").evaluate(kpo4))

BAGS: 3
VIALS: 11


In [7]:
# language=JSONata
res = jsonata.Jsonata("""$.[
            labeler_name,
            product_ndc,
            route,
            [active_ingredients]
      ]""").evaluate(kpo4)

rows = []

for labeler_name, product_ndc, route, ingredients in res:
      for ing in ingredients:
            s = parse_strength(ing["strength"])  # pint.Quantity

            strength_mg_ml = s.to("mg/mL")

            rows.append({
                  "labeler_name": labeler_name,
                  "product_ndc": product_ndc,
                  "name": ing["name"],
                  "route": route,
                  "strength": str(s),
                  "str/mL": float(strength_mg_ml.magnitude),
                  "str_unit": str((strength_mg_ml * reg.mL).units),
            })

df = pd.DataFrame(rows)
df

,labeler_name,product_ndc,name,route,strength,str/mL,str_unit
0,Caplin Steriles Limited,65145-195,"POTASSIUM PHOSPHATE, DIBASIC",INTRAVENOUS,236.0 mg / mL,236.00,mg
1,Caplin Steriles Limited,65145-195,"POTASSIUM PHOSPHATE, MONOBASIC",INTRAVENOUS,224.0 mg / mL,224.00,mg
2,Caplin Steriles Limited,65145-194,"POTASSIUM PHOSPHATE, DIBASIC",INTRAVENOUS,236.0 mg / mL,236.00,mg
3,Caplin Steriles Limited,65145-194,"POTASSIUM PHOSPHATE, MONOBASIC",INTRAVENOUS,224.0 mg / mL,224.00,mg
4,Caplin Steriles Limited,65145-196,"POTASSIUM PHOSPHATE, DIBASIC",INTRAVENOUS,236.0 mg / mL,236.00,mg
5,Caplin Steriles Limited,65145-196,"POTASSIUM PHOSPHATE, MONOBASIC",INTRAVENOUS,224.0 mg / mL,224.00,mg
6,"Fresenius Kabi USA, LLC",65219-656,DIBASIC POTASSIUM PHOSPHATE,INTRAVENOUS,11.8 mg / mL,11.80,mg
7,"Fresenius Kabi USA, LLC",65219-656,MONOBASIC POTASSIUM PHOSPHATE,INTRAVENOUS,11.2 mg / mL,11.20,mg
8,"Fresenius Kabi USA, LLC",65219-658,DIBASIC POTASSIUM PHOSPHATE,INTRAVENOUS,4.72 mg / mL,4.72,mg
9,"Fresenius Kabi USA, LLC",65219-658,MONOBASIC POTASSIUM PHOSPHATE,INTRAVENOUS,4.48 mg / mL,4.48,mg


### Sodium Chloride Bags

In [8]:
ns_bag = lookup_generic_name(ingredient_names=['sodium chloride'], dosage_form='INJECTION', max_active_ingredients=1,
                             fetch_rxcui=True)
df = pd.DataFrame(ns_bag)
df = df[df["product_type"] == 'BAG']
df['strength'] = df['active_ingredients'].apply(lambda ings: parse_strength(ings[0]['strength']) if ings else None)
df['strength'] = df['strength'].apply(
      lambda s: round(s.to("mg/mL"), 6) if s is not None else None
)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,packaged_as,package_type,package_size,strength
1,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-472,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-472-20', 'description'...","{'concept': [['1807634', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '500 mL', '...",BAG,500 mL,9.0 mg / mL
3,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-474,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-474-10', 'description'...","{'concept': [['1807639', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,9.0 mg / mL
4,"Fresenius Kabi USA, LLC",Sodium Chloride,SODIUM CHLORIDE,65219-328,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-328-50', 'description'...","{'concept': [['1807639', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,9.0 mg / mL
7,"Fresenius Medical Care de Mexico, S.A. de C.V.",Sodium Chloride,Sodium Chloride,46163-300,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '46163-300-10', 'description'...","{'concept': [['1807639', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,9.0 mg / mL
10,Baxter Healthcare Corporation,SODIUM CHLORIDE,sodium chloride,0338-9661,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-9661-60', 'description'...","{'concept': [['1807632', 'SCD']], 'drug': [['1...",other,BAG,None,NaN,NaN,9.0 mg / mL
11,Baxter Healthcare Corporation,SODIUM CHLORIDE,sodium chloride,0338-9662,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '0338-9662-35', 'description'...","{'concept': [['1807632', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '100 mL', '...",BAG,100 mL,9.0 mg / mL
12,"Fresenius Kabi USA, LLC",Sodium Chloride,SODIUM CHLORIDE,65219-432,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-432-85', 'description'...","{'concept': [['1807634', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '500 mL', '...",BAG,500 mL,9.0 mg / mL
13,Baxter Healthcare Corporation,SODIUM CHLORIDE,sodium chloride,0338-9663,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-9663-60', 'description'...","{'concept': [['1807632', 'SCD']], 'drug': [['1...",other,BAG,None,NaN,NaN,9.0 mg / mL
15,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-470,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-470-30', 'description'...","{'concept': [['1807633', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '250 mL', '...",BAG,250 mL,9.0 mg / mL
16,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-466,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-466-60', 'description'...","{'concept': [['1807631', 'SCD']], 'drug': [['1...",other,BAG,"{'packageCount': 1, 'packageSize': '50 mL', 'p...",BAG,50 mL,9.0 mg / mL


In [9]:
ns_bag = lookup_generic_name(ingredient_names=['sodium chloride'], dosage_form='INJECTION', max_active_ingredients=2)
df = pd.DataFrame(ns_bag)
df = df[df["product_type"] == 'BAG']
df['strength'] = df['active_ingredients'].apply(
      lambda ings: [round(parse_strength(ing['strength']), 5) for ing in ings] if ings else None)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,packaged_as,package_type,package_size,strength
1,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-472,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-472-20', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '500 mL', '...",BAG,500 mL,[9.0 mg / mL]
3,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-474,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-474-10', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,[9.0 mg / mL]
5,"Fresenius Kabi USA, LLC",Sodium Chloride,SODIUM CHLORIDE,65219-328,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '65219-328-50', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,[9.0 mg / mL]
7,"Fresenius Kabi USA, LLC",Potassium Chloride in Sodium Chloride,Sodium Chloride and Potassium Chloride,63323-686,"[{'name': 'POTASSIUM CHLORIDE', 'strength': '1...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '63323-686-10', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,"[1.5 mg / mL, 9.0 mg / mL]"
8,Baxter Healthcare Company,Potassium Chloride in Sodium Chloride,Potassium Chloride and Sodium Chloride,0338-0695,"[{'name': 'POTASSIUM CHLORIDE', 'strength': '3...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-0695-04', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,"[3.0 mg / mL, 9.0 mg / mL]"
9,"Fresenius Kabi USA, LLC",Potassium Chloride in Sodium Chloride,Sodium Chloride and Potassium Chloride,63323-688,"[{'name': 'POTASSIUM CHLORIDE', 'strength': '3...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '63323-688-10', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,"[3.0 mg / mL, 9.0 mg / mL]"
11,"Fresenius Medical Care de Mexico, S.A. de C.V.",Sodium Chloride,Sodium Chloride,46163-300,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '46163-300-10', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,[9.0 mg / mL]
14,Baxter Healthcare Corporation,SODIUM CHLORIDE,sodium chloride,0338-9661,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-9661-60', 'description'...",None,other,BAG,None,NaN,NaN,[0.009 g / mL]
16,Baxter Healthcare Company,Potassium Chloride in Sodium Chloride,Potassium Chloride and Sodium Chloride,0338-0704,"[{'name': 'POTASSIUM CHLORIDE', 'strength': '1...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-0704-34', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '1000 mL', ...",BAG,1000 mL,"[1.5 mg / mL, 4.5 mg / mL]"
17,Baxter Healthcare Corporation,SODIUM CHLORIDE,sodium chloride,0338-9662,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '0338-9662-35', 'description'...",None,other,BAG,"{'packageCount': 1, 'packageSize': '100 mL', '...",BAG,100 mL,[9.0 mg / mL]


### Ceftazidime-Avibactam (Avycaz)

In [10]:
avicaz = lookup_generic_name(ingredient_names=["avibactam", "ceftazidime"], fetch_rxcui=True)
avicaz = extract_product_type(avicaz)
df = pd.DataFrame(avicaz)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,packaged_as,package_type
0,"Allergan, Inc.",AVYCAZ,"ceftazidime, avibactam",0456-2700,"[{'name': 'AVIBACTAM SODIUM', 'strength': '.5 ...","POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0456-2700-10', 'description'...","{'concept': [['1603845', 'SBD']], 'drug': [['1...",other,"VIAL, SINGLE-DOSE",None,None
1,Fresenius Kabi iPSUM S.r.l.,NaN,"CEFTAZIDIME, AVIBACTAM SODIUM",66558-0212,"[{'name': 'AVIBACTAM SODIUM', 'strength': '.2 ...",POWDER,None,"[{'package_ndc': '66558-0212-0', 'description'...",{},other,CONTAINER,None,None


In [11]:
vaso = lookup_generic_name('vasopressin', dosage_form='INJECTION', fetch_rxcui=True)
data = parse_strength_in_context(vaso)
df = pd.DataFrame(data)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,packaged_as,package_type,package_size
0,"Par Health USA, LLC",Vasostrict,Vasopressin,42023-190,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '42023-190-01', 'description'...","{'concept': [['1593738', 'SBD']], 'drug': [['3...",generic_name,VIAL,"{'packageCount': 1, 'packageSize': '10 mL', 'p...",VIAL,10 mL
1,Gland Pharma Limited,Vasopressin,Vasopressin,68083-663,"[{'name': 'VASOPRESSIN', 'strength': 0.4 unit ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '68083-663-10', 'description'...","{'concept': [['2399907', 'SCD']], 'drug': [['2...",generic_name,VIAL,"{'packageCount': 1, 'packageSize': '100 mL', '...",VIAL,100 mL
2,"Par Health USA, LLC",Vasostrict,Vasopressin,42023-219,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '42023-219-10', 'description'...","{'concept': [['2399909', 'SBD']], 'drug': [['2...",generic_name,VIAL,"{'packageCount': 1, 'packageSize': '100 mL', '...",VIAL,100 mL
3,"Par Health USA, LLC",Vasostrict,Vasopressin,42023-237,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '42023-237-10', 'description'...","{'concept': [['2591431', 'SBD']], 'drug': [['2...",generic_name,VIAL,"{'packageCount': 1, 'packageSize': '100 mL', '...",VIAL,100 mL
4,"Amphastar Pharmaceuticals, Inc.",Vasopressin,Vasopressin,0548-9701,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '0548-9701-00', 'description'...","{'concept': [['2103182', 'SCD']], 'drug': [['2...",generic_name,VIAL,"{'packageCount': 1, 'packageSize': '1 mL', 'pa...",VIAL,1 mL
5,"Fresenius Kabi USA, LLC",VASOPRESSIN,VASOPRESSIN,63323-930,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '63323-930-01', 'description'...","{'concept': [['2103182', 'SCD']], 'drug': [['2...",generic_name,"VIAL, SINGLE-DOSE","{'packageCount': 1, 'packageSize': '1 mL', 'pa...","VIAL, SINGLE-DOSE",1 mL
6,"Dr. Reddy's Laboratories, Inc.",vasopressin,Vasopressin,43598-914,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '43598-914-06', 'description'...","{'concept': [['2103182', 'SCD']], 'drug': [['2...",generic_name,VIAL,"{'packageCount': 1, 'packageSize': '1 mL', 'pa...",VIAL,1 mL
7,"American Regent, Inc.",Vasopressin,Vasopressin,0517-1020,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0517-1020-25', 'description'...","{'concept': [['2103182', 'SCD']], 'drug': [['2...",generic_name,"VIAL, SINGLE-DOSE","{'packageCount': 1, 'packageSize': '1 mL', 'pa...","VIAL, SINGLE-DOSE",1 mL
8,"Medical Purchasing Solutions, LLC",Vasostrict,Vasopressin,71872-7264,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '71872-7264-1', 'description'...","{'concept': [['2103184', 'SBD']], 'drug': [['2...",generic_name,VIAL,"{'packageCount': 1, 'packageSize': '1 mL', 'pa...",VIAL,1 mL
9,"American Regent, Inc.",Vasopressin,Vasopressin,0517-1030,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0517-1030-01', 'description'...","{'concept': [['313578', 'SCD']], 'drug': [['31...",generic_name,"VIAL, MULTI-DOSE","{'packageCount': 1, 'packageSize': '10 mL', 'p...","VIAL, MULTI-DOSE",10 mL


In [12]:
unasyn = lookup_generic_name("ampicillin")
unasyn = extract_product_type(unasyn)
df = pd.DataFrame(unasyn)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,packaged_as,package_type,package_size
0,Sagent Pharmaceuticals,ampicillin,ampicillin,25021-136,"[{'name': 'AMPICILLIN SODIUM', 'strength': '1 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '25021-136-10', 'description'...",None,generic_name,VIAL,None,NaN,NaN
1,Sagent Pharmaceuticals,ampicillin,ampicillin,25021-137,"[{'name': 'AMPICILLIN SODIUM', 'strength': '2 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '25021-137-20', 'description'...",None,generic_name,VIAL,None,NaN,NaN
2,Aurobindo Pharma Limited,Ampicillin,Ampicillin,59651-551,"[{'name': 'AMPICILLIN TRIHYDRATE', 'strength':...",CAPSULE,[ORAL],"[{'package_ndc': '59651-551-01', 'description'...",None,generic_name,CAPSULE,None,NaN,NaN
3,Aurobindo Pharma Limited,Ampicillin,Ampicillin,59651-550,"[{'name': 'AMPICILLIN TRIHYDRATE', 'strength':...",CAPSULE,[ORAL],"[{'package_ndc': '59651-550-01', 'description'...",None,generic_name,CAPSULE,None,NaN,NaN
4,"Civica, Inc.",AMPICILLIN,Ampicillin,72572-017,"[{'name': 'AMPICILLIN SODIUM', 'strength': '2 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '72572-017-10', 'description'...",None,generic_name,"VIAL, GLASS",None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,"Henry Schein, Inc.",AMPICILLIN,AMPICILLIN SODIUM,0404-9818,"[{'name': 'AMPICILLIN SODIUM', 'strength': '2 ...","INJECTION, POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0404-9818-99', 'description'...",None,generic_name,VIAL,None,NaN,NaN
96,Aurobindo Pharma Limited,NaN,Ampicillin Sodium (Sterile),65862-282,"[{'name': 'AMPICILLIN SODIUM', 'strength': '50...",POWDER,None,"[{'package_ndc': '65862-282-37', 'description'...",None,generic_name,CANISTER,None,NaN,NaN
97,Meitheal Pharmaceuticals Inc.,Ampicillin and Sulbactam,Ampicillin sodium and Sulbactam sodium,71288-032,"[{'name': 'AMPICILLIN SODIUM', 'strength': '2 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '71288-032-21', 'description'...",None,generic_name,"VIAL, SINGLE-DOSE",None,NaN,NaN
98,EUMEDICA PHARMACEUTICALS INDUSTRIES SOCIEDAD L...,NaN,Ampicillin and Sulbactam Sodium,84344-002,"[{'name': 'AMPICILLIN SODIUM', 'strength': '33...",POWDER,None,"[{'package_ndc': '84344-002-01', 'description'...",None,generic_name,BAG,None,NaN,NaN


In [21]:
bupiv = lookup_generic_name(ingredient_names=['bupivacaine'], dosage_form="INJECTION", fetch_rxcui=True)
JSON(bupiv)

<IPython.core.display.JSON object>

In [13]:
result = jsonata.Jsonata("$.[active_ingredients.[name, strength]]").evaluate(vaso)

In [14]:
dxs = jsonata.Jsonata("**.description").evaluate(vaso)
for dx in dxs:
      print(dx)
print(re.findall(r'\b[A-Z|\s|,|\-]{3,}', dx)[-1])

1 VIAL in 1 CARTON (42023-190-01)  / 10 mL in 1 VIAL
10 VIAL in 1 CARTON (68083-663-10)  / 100 mL in 1 VIAL
10 VIAL in 1 CARTON (42023-219-10)  / 100 mL in 1 VIAL (42023-219-01)
10 VIAL in 1 CARTON (42023-237-10)  / 100 mL in 1 VIAL (42023-237-01)
1 VIAL in 1 CARTON (0548-9701-00)  / 1 mL in 1 VIAL
25 VIAL, SINGLE-DOSE in 1 TRAY (63323-930-01)  / 1 mL in 1 VIAL, SINGLE-DOSE (63323-930-00)
25 VIAL in 1 CARTON (43598-914-06)  / 1 mL in 1 VIAL (43598-914-11)
25 VIAL in 1 CARTON (43598-914-25)  / 1 mL in 1 VIAL (43598-914-11)
25 VIAL, SINGLE-DOSE in 1 CARTON (0517-1020-25)  / 1 mL in 1 VIAL, SINGLE-DOSE (0517-1020-01)
1 VIAL in 1 BAG (71872-7264-1)  / 1 mL in 1 VIAL
1 VIAL, MULTI-DOSE in 1 CARTON (0517-1030-01)  / 10 mL in 1 VIAL, MULTI-DOSE
1 mL in 1 VIAL, MULTI-DOSE (51662-1623-1)
1 VIAL, MULTI-DOSE in 1 POUCH (51662-1623-2)  / 1 mL in 1 VIAL, MULTI-DOSE
1 mL in 1 VIAL (51662-1314-1)
1 VIAL in 1 POUCH (51662-1314-2)  / 1 mL in 1 VIAL
25 VIAL in 1 CARTON (25021-474-01)  / 1 mL in 1 VIAL
2

In [15]:
# language=JSONata
res = jsonata.Jsonata("""$.{labeler_name:
                        [active_ingredients.{
                              'name': name,
                              'strength': strength,
                              'product_type': product_type,
                              'product_ndc': %.product_ndc
                        },
                        [**.description]]
                  }""").evaluate(vaso)
# data = parse_strength_in_context(res)
JSON(res)

<IPython.core.display.JSON object>

### Penicillin G

In [22]:
pen_g = lookup_generic_name("penicillin g", fetch_rxcui=True)
df = pd.DataFrame(pen_g)
df['api'] = df['active_ingredients'].apply(lambda ings: ings[0]['name'] if ings else None)
df['api_strength'] = df['active_ingredients'].apply(lambda ings: ings[0]['strength'] if ings else None)
df['api_2'] = df['active_ingredients'].apply(lambda ings: ings[1]['name'] if len(ings) > 1 else None)
df['api_2_strength'] = df['active_ingredients'].apply(lambda ings: ings[1]['strength'] if len(ings) > 1 else None)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,packaged_as,package_type,package_size,api,api_strength,api_2,api_2_strength
0,Baxter Healthcare Corporation,PENICILLIN G POTASSIUM,PENICILLIN G,0338-1021,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-1021-41', 'description'...","{'concept': [['207390', 'SCD']], 'drug': [['20...",generic_name,BAG,None,NaN,NaN,PENICILLIN G POTASSIUM,1000000 [iU]/50mL,NaN,NaN
1,Baxter Healthcare Corporation,PENICILLIN G POTASSIUM,PENICILLIN G,0338-1023,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-1023-41', 'description'...","{'concept': [['204466', 'SCD']], 'drug': [['20...",generic_name,BAG,None,NaN,NaN,PENICILLIN G POTASSIUM,2000000 [iU]/50mL,NaN,NaN
2,Baxter Healthcare Corporation,PENICILLIN G POTASSIUM,PENICILLIN G,0338-1025,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-1025-41', 'description'...","{'concept': [['207391', 'SCD']], 'drug': [['20...",generic_name,BAG,None,NaN,NaN,PENICILLIN G POTASSIUM,3000000 [iU]/50mL,NaN,NaN
3,Roerig,Pfizerpen,penicillin G potassium,0049-0530,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0049-0530-22', 'description'...","{'concept': [['995906', 'SBD']], 'drug': [['99...",generic_name,VIAL,None,NaN,NaN,PENICILLIN G POTASSIUM,20000000 [iU]/1,NaN,NaN
4,"NorthStar Rx, LLC",Penicillin G Potassium,Penicillin G Potassium,72603-345,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '72603-345-01', 'description'...","{'concept': [['863538', 'SCD']], 'drug': [['99...",generic_name,VIAL,None,NaN,NaN,PENICILLIN G POTASSIUM,20000000 [iU]/1,NaN,NaN
5,Sagent Pharmaceuticals,Penicillin G Potassium,penicillin g potassium,25021-153,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '25021-153-20', 'description'...","{'concept': [['863538', 'SCD']], 'drug': [['99...",generic_name,VIAL,None,NaN,NaN,PENICILLIN G POTASSIUM,5000000 [iU]/1,NaN,NaN
6,Sagent Pharmaceuticals,Penicillin G Potassium,penicillin g potassium,25021-154,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '25021-154-51', 'description'...","{'concept': [['863538', 'SCD']], 'drug': [['99...",generic_name,VIAL,None,NaN,NaN,PENICILLIN G POTASSIUM,20000000 [iU]/1,NaN,NaN
7,Sandoz Inc,Penicillin G Potassium,Penicillin G Potassium,0781-6135,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '0781-6135-95', 'description'...","{'concept': [['863538', 'SCD']], 'drug': [['99...",generic_name,VIAL,None,NaN,NaN,PENICILLIN G POTASSIUM,5000000 [iU]/1,NaN,NaN
8,Sandoz Inc,Penicillin G Potassium,Penicillin G Potassium,0781-6136,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '0781-6136-94', 'description'...","{'concept': [['863538', 'SCD']], 'drug': [['99...",generic_name,VIAL,None,NaN,NaN,PENICILLIN G POTASSIUM,20000000 [iU]/1,NaN,NaN
9,Avenacy Inc.,Penicillin G Potassium,penicillin g potassium,83634-103,"[{'name': 'PENICILLIN G POTASSIUM', 'strength'...","INJECTION, POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '83634-103-51', 'description'...","{'concept': [['863538', 'SCD']], 'drug': [['99...",generic_name,VIAL,None,NaN,NaN,PENICILLIN G POTASSIUM,20000000 [iU]/1,NaN,NaN


In [ ]:
df[df['labeler_name'].str.contains('Roerig', na=False)]

# Filterable Drugs

In [ ]:
def get_inline_filter_drugs():
      url = r"http://pmc.ncbi.nlm.nih.gov/articles/PMC11907493/table/table1-00185787251324867"
      df = pd.read_html(url)[0]
      # remove NBSP chars
      df['Drug'] = df['Drug'].str.replace(r'\xa0', ' ')
      df['brand'] = df['Drug'].str.findall(r'\((.*?)\)')
      return df


filterable_drugs = get_inline_filter_drugs()
filterable_drugs

# RXCUI